In [ ]:
!python -m pip install numpy pandas \
    qiskit-aer qiskit-algorithms qiskit-machine-learning qiskit-ibm-runtime \
    pylatexenc ucimlrepo \
    xgboost catboost seaborn libsvm-official \
    jinja2 \
    scikit-optimize \
    tensorflow

In [ ]:
import sys
import os
from pathlib import Path

def find_project_root(marker=".gitignore"):
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"'{marker}' tidak ditemukan")

project_root = find_project_root()
print(f"Project root: {project_root}")

sys.path.append(os.path.abspath(project_root))

from utils.prepare_data import prepare_data

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)

In [ ]:
import warnings
from qiskit.transpiler import generate_preset_pass_manager
warnings.filterwarnings('ignore')

In [ ]:
import sys
import importlib

def reload_package(package_name):
    modules_to_reload = [
        name for name in sys.modules
        if name.startswith(package_name)
    ]

    for name in sorted(modules_to_reload, reverse=True):
        importlib.reload(sys.modules[name])

# Usage
reload_package("model")
reload_package("utils")

In [ ]:
dataset_path = os.path.join(project_root, "dataset", "Dataset_TehHijau.csv")
feature_cols = [
        "MQ3", 
        "TGS822", 
        "TGS2602", 
        "MQ5", 
        "MQ138", 
        "TGS2620", 
        "TGS813", 
        "TGS2600", 
        "TGS2611", 
        "TGS2603",
        "Humidity",
        "Celsius",
    ]
target_cols = "Kategori"

quantum_kernel_types = [
    # 'full', 
    'linear', 
    # 'circular', 
    # 'pauli_x', 
    # 'pauli_y', 
    # 'pauli_z'
]
kernel_types = [
    'linear', 
    # 'poly', 
    # 'rbf', 
    # 'sigmoid'
]

In [ ]:
import pandas as pd
data = pd.read_csv(dataset_path)
data.head(10)

In [ ]:
data.describe()

In [ ]:
import matplotlib.pyplot as plt
data['Kategori'].hist()
plt.xlabel('Kategori')
plt.ylabel('Count')
plt.title('Histogram for the Kategori')

In [ ]:
X = data[feature_cols]
y = data[target_cols]

print(X.shape, y.shape)

In [ ]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

In [ ]:
from sklearn.decomposition import PCA
import numpy as np
import matplotlib.pyplot as plt
def plot_pca_variance(X, n_components, threshold=0.95):
  pca = PCA(n_components=n_components)
  pca.fit(X)

  cumvar = np.cumsum(pca.explained_variance_ratio_)
  n_optimal = np.argmax(cumvar >= threshold) + 1

  plt.figure(figsize=(8, 5))
  plt.plot(range(1, n_components + 1), cumvar, marker='o', color='blue')
  plt.axhline(y=threshold, color='gray', linestyle='--', linewidth=1)
  plt.xlabel('Number of Principal Components')
  plt.ylabel('Cumulative Explained Variance')
  plt.xticks(range(1, n_components + 1))
  plt.grid(True)
  plt.tight_layout()
  plt.show()

  print(f'Optimal number of components to retain {threshold*100:.0f}% variance: {n_optimal}')

  return n_optimal

In [ ]:
n_optimal = plot_pca_variance(X, n_components=X.shape[1], threshold=0.92)

In [ ]:
import time
from pathlib import Path
from datetime import datetime

all_best = []  # collects best_result from every model
_iter_t0 = None

# Logger setup
log_dir = Path("./results/logs/gs/classical/all")
log_dir.mkdir(parents=True, exist_ok=True)

_current_log_path = None

def setup_logger(name):
  """Panggil di awal tiap section. name contoh: 'svc_linear', 'xgb_gbtree'."""
  global _current_log_path
  _current_log_path = log_dir / f"{name}.log"
  # truncate file kalau sudah ada, biar tiap run fresh
  _current_log_path.write_text("")
  log(f"📝 Log: {_current_log_path}")
  log(f"🕒 Started: {datetime.now().isoformat(timespec='seconds')}")

def log(msg=""):
  """Print ke console dan append ke file log section aktif."""
  print(msg)
  if _current_log_path is not None:
    with open(_current_log_path, "a", encoding="utf-8") as f:
      f.write(str(msg) + "\n")


In [ ]:
# ────────────────────────────────────────────────────────────
# TensorFlow / Keras setup: deterministic seed + GPU memory growth
# ────────────────────────────────────────────────────────────
import os
# os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')  # silence TF info logs

import numpy as np
import tensorflow as tf
from tensorflow import keras

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Optional: allow GPU memory growth (avoid OOM on shared GPUs)
_gpus = tf.config.list_physical_devices('GPU')
for _g in _gpus:
    try:
        tf.config.experimental.set_memory_growth(_g, True)
    except Exception:
        pass

print(f'TF {tf.__version__} | GPUs detected: {len(_gpus)}')
n_classes = int(len(np.unique(y)))
print(f'n_classes = {n_classes}')

# 1. MLP

In [ ]:
# Search Space Configuration — MLP
import numpy as np

search_space = {
  'hidden_units' : [(64, 32), (128, 64, 32)],
  'dropout'      : [0.1, 0.3],
  'learning_rate': [1e-3, 1e-4],
  'batch_size'   : [32, 64],
  'epochs'       : [100],
}

param_keys = search_space.keys()
param_vals = search_space.values()

In [ ]:
setup_logger("mlp")
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from itertools import product
from sklearn.metrics import (
  accuracy_score, f1_score, roc_auc_score,
  average_precision_score, matthews_corrcoef,
  precision_score, recall_score
)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

state = 42
n_splits = 5

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=state)

search_space_sizes = {k: len(v) for k, v in search_space.items()}
total_configs = int(np.prod([len(v) for v in param_vals]))
total_fits = total_configs * skf.get_n_splits()
space_str = " × ".join([f"{search_space_sizes[k]} {k}" for k in search_space_sizes])

log(f"🔬 Search space: {space_str} =  {total_configs} configs × {skf.get_n_splits()} folds = {total_fits} fits")
log("   Scoring criterion: (AUROC + PRAUC + Accuracy) / 3")

def build_mlp(input_dim, n_classes, hidden_units, dropout, learning_rate):
    tf.keras.utils.set_random_seed(42)
    inputs = keras.Input(shape=(input_dim,))
    x = inputs
    for h in hidden_units:
        x = layers.Dense(h, activation='relu',
                         kernel_initializer='he_normal')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(n_classes, activation='softmax')(x)
    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model

best_score = -np.inf
best_result = None
results = []
for i, comb in enumerate(product(*param_vals)):
  params = dict(zip(param_keys, comb))
  tag = " | ".join(f"{k}={v}" for k, v in params.items())
  log(f"\n  ▶ [{i+1}/{total_configs}] {tag}")
  _iter_t0 = time.perf_counter()

  accs, f1s, rocs, pras, precs, recs = [], [], [], [], [], []
  y_val_all, y_pred_all = [], []
  for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    # Scale + PCA (same pipeline philosophy as classical models)
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s   = scaler.transform(X_val)

    pca = PCA(n_components=n_optimal)
    X_train_p = pca.fit_transform(X_train_s)
    X_val_p   = pca.transform(X_val_s)

    # Build & train
    keras.backend.clear_session()
    model = build_mlp(
        input_dim=X_train_p.shape[1],
        n_classes=n_classes,
        hidden_units=params['hidden_units'],
        dropout=params['dropout'],
        learning_rate=params['learning_rate'],
    )
    es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=15,
                                       restore_best_weights=True, verbose=0)
    model.fit(
        X_train_p, y_train,
        validation_data=(X_val_p, y_val),
        epochs=params['epochs'],
        batch_size=params['batch_size'],
        verbose=0,
        callbacks=[es],
    )

    y_prob = model.predict(X_val_p, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)

    y_val_all.extend(y_val)
    y_pred_all.extend(y_pred)

    acc  = accuracy_score(y_val, y_pred)
    f1   = f1_score(y_val, y_pred, average='weighted')
    roc  = roc_auc_score(y_val, y_prob, average='weighted', multi_class='ovr')
    pra  = average_precision_score(y_val, y_prob, average='weighted')
    prec = precision_score(y_val, y_pred, average='weighted')
    rec  = recall_score(y_val, y_pred, average='weighted')

    accs.append(acc); f1s.append(f1); rocs.append(roc)
    pras.append(pra); precs.append(prec); recs.append(rec)

    log(f"    F{fold} → Acc={acc:.4f} | Prec={prec:.4f} | Rec={rec:.4f} | F1={f1:.4f} | AUROC={roc:.4f} | PRAUC={pra:.4f}")

    results.append({
      "tag": tag, **params,
      "fold": fold, "accuracy": acc, "precision": prec,
      "recall": rec, "F1": f1, "auroc": roc, "prauc": pra,
    })

  acc_mean,  acc_std  = np.mean(accs),  np.std(accs)
  prec_mean, prec_std = np.mean(precs), np.std(precs)
  rec_mean,  rec_std  = np.mean(recs),  np.std(recs)
  f1_mean,   f1_std   = np.mean(f1s),   np.std(f1s)
  roc_mean,  roc_std  = np.mean(rocs),  np.std(rocs)
  pra_mean,  pra_std  = np.mean(pras),  np.std(pras)
  mcc = matthews_corrcoef(y_val_all, y_pred_all)

  composite = (roc_mean + pra_mean + acc_mean) / 3

  log(
    f"  ✅  Acc:{acc_mean:.4f}±{acc_std:.4f} | "
    f"Precision:{prec_mean:.4f}±{prec_std:.4f}  |"
    f"Recall:{rec_mean:.4f}±{rec_std:.4f} |"
    f"F1:{f1_mean:.4f}±{f1_std:.4f} | "
    f"AUROC:{roc_mean:.4f}±{roc_std:.4f} | "
    f"PRAUC:{pra_mean:.4f}±{pra_std:.4f} | "
    f"MCC:{mcc:.4f} | "
    f"Composite:{composite:.4f}"
  )

  iter_time = time.perf_counter() - _iter_t0
  if composite > best_score:
    best_score = composite
    best_result = {
        'tag': tag,
        'composite': composite,
        'roc':  f"{roc_mean:.4f}±{roc_std:.4f}",
        'pra':  f"{pra_mean:.4f}±{pra_std:.4f}",
        'acc':  f"{acc_mean:.4f}±{acc_std:.4f}",
        'f1':   f"{f1_mean:.4f}±{f1_std:.4f}",
        'prec': f"{prec_mean:.4f}±{prec_std:.4f}",
        'rec':  f"{rec_mean:.4f}±{rec_std:.4f}",
        'params': params,
        'execution_time': iter_time,
    }

log(f"\n🏆 Best config : {best_result['tag']}")
log(
    f"   Composite   : {best_result['composite']} "
    f"(AUROC={best_result['roc']} | "
    f"PRAUC={best_result['pra']} | "
    f"Acc={best_result['acc']} |"
    f"Prec={best_result['prec']} |"
    f"Rec={best_result['rec']}) |"
)

In [ ]:
import pandas as pd
import os

dirpath = "./results/"
os.makedirs(dirpath, exist_ok=True)

df = pd.DataFrame(results)
filename = dirpath + "mlp.csv"
df.to_csv(filename, index=False)

log(f"✅ Saved: {filename}")

In [ ]:
# ── Collect best result ──
best_result['model'] = "MLP"
all_best.append(dict(best_result))
log(f"✅ [MLP] recorded | Exec. time: {best_result['execution_time']:.1f}s")

# 2. 1D CNN

In [ ]:
# Search Space Configuration — 1D CNN
import numpy as np

search_space = {
  'filters'      : [(32, 64), (64, 128)],
  'kernel_size'  : [2, 3],
  'dropout'      : [0.1, 0.3],
  'learning_rate': [1e-3, 1e-4],
  'batch_size'   : [32],
  'epochs'       : [100],
}

param_keys = search_space.keys()
param_vals = search_space.values()

In [ ]:
setup_logger("cnn1d")
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from itertools import product
from sklearn.metrics import (
  accuracy_score, f1_score, roc_auc_score,
  average_precision_score, matthews_corrcoef,
  precision_score, recall_score
)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

state = 42
n_splits = 5

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=state)

search_space_sizes = {k: len(v) for k, v in search_space.items()}
total_configs = int(np.prod([len(v) for v in param_vals]))
total_fits = total_configs * skf.get_n_splits()
space_str = " × ".join([f"{search_space_sizes[k]} {k}" for k in search_space_sizes])

log(f"🔬 Search space: {space_str} =  {total_configs} configs × {skf.get_n_splits()} folds = {total_fits} fits")
log("   Scoring criterion: (AUROC + PRAUC + Accuracy) / 3")

def build_cnn1d(input_len, n_classes, filters, kernel_size, dropout, learning_rate):
    tf.keras.utils.set_random_seed(42)
    inputs = keras.Input(shape=(input_len, 1))
    x = inputs
    for f in filters:
        # 'same' padding so small input_len doesn't shrink to zero
        x = layers.Conv1D(f, kernel_size=kernel_size, padding='same',
                          activation='relu',
                          kernel_initializer='he_normal')(x)
        x = layers.BatchNormalization()(x)
        # MaxPool only if length still > 1
        if x.shape[1] is not None and x.shape[1] > 1:
            x = layers.MaxPooling1D(pool_size=2)(x)
        x = layers.Dropout(dropout)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(n_classes, activation='softmax')(x)
    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model

best_score = -np.inf
best_result = None
results = []
for i, comb in enumerate(product(*param_vals)):
  params = dict(zip(param_keys, comb))
  tag = " | ".join(f"{k}={v}" for k, v in params.items())
  log(f"\n  ▶ [{i+1}/{total_configs}] {tag}")
  _iter_t0 = time.perf_counter()

  accs, f1s, rocs, pras, precs, recs = [], [], [], [], [], []
  y_val_all, y_pred_all = [], []
  for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    # Scale + PCA, then reshape to (samples, features, 1) for Conv1D
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s   = scaler.transform(X_val)

    pca = PCA(n_components=n_optimal)
    X_train_p = pca.fit_transform(X_train_s)
    X_val_p   = pca.transform(X_val_s)

    X_train_c = X_train_p[..., np.newaxis]
    X_val_c   = X_val_p[..., np.newaxis]

    keras.backend.clear_session()
    model = build_cnn1d(
        input_len=X_train_c.shape[1],
        n_classes=n_classes,
        filters=params['filters'],
        kernel_size=params['kernel_size'],
        dropout=params['dropout'],
        learning_rate=params['learning_rate'],
    )
    es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=15,
                                       restore_best_weights=True, verbose=0)
    model.fit(
        X_train_c, y_train,
        validation_data=(X_val_c, y_val),
        epochs=params['epochs'],
        batch_size=params['batch_size'],
        verbose=0,
        callbacks=[es],
    )

    y_prob = model.predict(X_val_c, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)

    y_val_all.extend(y_val)
    y_pred_all.extend(y_pred)

    acc  = accuracy_score(y_val, y_pred)
    f1   = f1_score(y_val, y_pred, average='weighted')
    roc  = roc_auc_score(y_val, y_prob, average='weighted', multi_class='ovr')
    pra  = average_precision_score(y_val, y_prob, average='weighted')
    prec = precision_score(y_val, y_pred, average='weighted')
    rec  = recall_score(y_val, y_pred, average='weighted')

    accs.append(acc); f1s.append(f1); rocs.append(roc)
    pras.append(pra); precs.append(prec); recs.append(rec)

    log(f"    F{fold} → Acc={acc:.4f} | Prec={prec:.4f} | Rec={rec:.4f} | F1={f1:.4f} | AUROC={roc:.4f} | PRAUC={pra:.4f}")

    results.append({
      "tag": tag, **params,
      "fold": fold, "accuracy": acc, "precision": prec,
      "recall": rec, "F1": f1, "auroc": roc, "prauc": pra,
    })

  acc_mean,  acc_std  = np.mean(accs),  np.std(accs)
  prec_mean, prec_std = np.mean(precs), np.std(precs)
  rec_mean,  rec_std  = np.mean(recs),  np.std(recs)
  f1_mean,   f1_std   = np.mean(f1s),   np.std(f1s)
  roc_mean,  roc_std  = np.mean(rocs),  np.std(rocs)
  pra_mean,  pra_std  = np.mean(pras),  np.std(pras)
  mcc = matthews_corrcoef(y_val_all, y_pred_all)

  composite = (roc_mean + pra_mean + acc_mean) / 3

  log(
    f"  ✅  Acc:{acc_mean:.4f}±{acc_std:.4f} | "
    f"Precision:{prec_mean:.4f}±{prec_std:.4f}  |"
    f"Recall:{rec_mean:.4f}±{rec_std:.4f} |"
    f"F1:{f1_mean:.4f}±{f1_std:.4f} | "
    f"AUROC:{roc_mean:.4f}±{roc_std:.4f} | "
    f"PRAUC:{pra_mean:.4f}±{pra_std:.4f} | "
    f"MCC:{mcc:.4f} | "
    f"Composite:{composite:.4f}"
  )

  iter_time = time.perf_counter() - _iter_t0
  if composite > best_score:
    best_score = composite
    best_result = {
        'tag': tag,
        'composite': composite,
        'roc':  f"{roc_mean:.4f}±{roc_std:.4f}",
        'pra':  f"{pra_mean:.4f}±{pra_std:.4f}",
        'acc':  f"{acc_mean:.4f}±{acc_std:.4f}",
        'f1':   f"{f1_mean:.4f}±{f1_std:.4f}",
        'prec': f"{prec_mean:.4f}±{prec_std:.4f}",
        'rec':  f"{rec_mean:.4f}±{rec_std:.4f}",
        'params': params,
        'execution_time': iter_time,
    }

log(f"\n🏆 Best config : {best_result['tag']}")
log(
    f"   Composite   : {best_result['composite']} "
    f"(AUROC={best_result['roc']} | "
    f"PRAUC={best_result['pra']} | "
    f"Acc={best_result['acc']} |"
    f"Prec={best_result['prec']} |"
    f"Rec={best_result['rec']}) |"
)

In [ ]:
import pandas as pd
import os

dirpath = "./results/"
os.makedirs(dirpath, exist_ok=True)

df = pd.DataFrame(results)
filename = dirpath + "cnn1d.csv"
df.to_csv(filename, index=False)

log(f"✅ Saved: {filename}")

In [ ]:
# ── Collect best result ──
best_result['model'] = "1D CNN"
all_best.append(dict(best_result))
log(f"✅ [1D CNN] recorded | Exec. time: {best_result['execution_time']:.1f}s")

---
## 📊 Tabel Evaluasi Akhir

In [ ]:
# ════════════════════════════════════════════════════════════
# 📊 TABEL EVALUASI AKHIR – Deep Learning Models
# ════════════════════════════════════════════════════════════
import pandas as pd

def build_eval_table(all_best):
    rows = []
    for r in all_best:
        params_str = ' | '.join(f'{k}={v}' for k, v in r.get('params', {}).items())
        rows.append({
            'Model'          : r['model'],
            'Accuracy'       : r['acc'],
            'Precision'      : r['prec'],
            'Recall'         : r['rec'],
            'F1-Score'       : r['f1'],
            'ROC-AUC'        : r['roc'],
            'PR-AUC'         : r['pra'],
            'Exec. Time (s)' : round(r['execution_time'], 2) if 'execution_time' in r else 'N/A',
            'Best Params'    : params_str,
        })
    df = pd.DataFrame(rows)
    df = df.sort_values('ROC-AUC', ascending=False).reset_index(drop=True)
    df.index += 1
    return df

pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', '{:.4f}'.format)

eval_df = build_eval_table(all_best)
display(eval_df)

In [ ]:
import os
os.makedirs('./results', exist_ok=True)
eval_df.to_csv('./results/eval_dl_final.csv', index=True)
print('✅ Saved: ./results/eval_dl_final.csv')